In [26]:
import pandas as pd

In [58]:
data = pd.read_csv('data/training.csv')

In [28]:
data.isna().sum().sum()

np.int64(0)

U datasetu nema null vrednosti.

In [29]:
data.select_dtypes(include='str')

,PIDN,Depth
0,XNhoFZW5,Topsoil
1,9XNspFTd,Subsoil
2,WDId41qG,Topsoil
3,JrrJf1mN,Subsoil
4,ZoIitegA,Topsoil
...,...,...
1152,bdcNNrbi,Topsoil
1153,6HBVKZwh,Subsoil
1154,5eLY5nw7,Topsoil
1155,gsSGXhX6,Subsoil


Izbacujemo PDIN koji predstavlja identifikator uzorka

In [59]:
data.drop('PIDN', inplace=True, axis=1)

Pretvaramo Depth obelezje u numericke vrednosti

In [60]:
data['Depth'] = data['Depth'].replace({'Topsoil' : 0 , 'Subsoil' : 1})

## Provera korelacija

In [32]:
#import matplotlib as plt
#import seaborn as sb
#
#def korelaciona_matrica(name):
#    corr_matrix = data.corr(numeric_only=True).round(2)
#    plt.rcParams['figure.figsize']=[100,100]
#    sb.heatmap(corr_matrix, annot=True, cmap='coolwarm')
#    plt.title('Korelaciona matrica')
#    plt.savefig(name)
#    plt.show()

In [33]:
#korelaciona_matrica('korelaciona_matrica_1')

racunanje korelacije mi je ubilo operativni sistem

## Outlier-i

In [63]:
from scipy.stats import normaltest

norm = []
columns = data.select_dtypes(include='number').columns.tolist()
for column in columns:
    p = normaltest(data[column])
    if p.pvalue < 0.05:
        norm.append(False)
    else:
        norm.append(True)

print(pd.DataFrame(norm).sum(), data.shape)

0    418
dtype: int64 (1157, 3599)


Od 3599 kolona svega 418 ima normalnu raspodelu, pa ne mozemo koristiti z_score za uklanjanje outlier-a

In [66]:
data_numeric = data.select_dtypes(include='number')

data_bez_outliera = data

q1 = data_numeric.quantile(0.25)
q3 = data_numeric.quantile(0.75)
iqr = q3 - q1

columns = data_numeric.columns
for column in columns:
    if (column == 'Ca') | (column == 'P') | (column == 'pH') | (column == 'SOC') | (column == 'Sand'):
        continue
    data_bez_outliera = data_bez_outliera[(data_bez_outliera[column] >= q1[column] - 1.5 * iqr[column]) & (data_bez_outliera[column] <= q3[column] + 1.5 * iqr[column])]

In [68]:
data_bez_outliera.shape

(715, 3599)

Izbacivanjem outliera izgubili smo preveliku kolicinu podataka, pa cemo ih clippovati

In [72]:
data_numeric = data.select_dtypes(include='number')

data_clipped = data

q1 = data_numeric.quantile(0.25)
q3 = data_numeric.quantile(0.75)
iqr = q3 - q1

columns = data_numeric.columns
for column in columns:
    if (column == 'Ca') | (column == 'P') | (column == 'pH') | (column == 'SOC') | (column == 'Sand'):
        continue
    min = q1[column] - 1.5 * iqr[column]
    max = q3[column] + 1.5 * iqr[column]
    data_clipped[column] = data_clipped[column].clip(upper=max, lower=min)

## Modeli

In [36]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

model_lin_reg = LinearRegression(n_jobs=-1)
pipeline_lin_reg = Pipeline([
    ('scaler', StandardScaler()),
    ('model_lin_reg', model_lin_reg),
])
grid_params = {
    'scaler' : [None, StandardScaler()],
    'model_lin_reg__fit_intercept' : [False, True],
}

grid_lin_reg = GridSearchCV(pipeline_lin_reg, grid_params, n_jobs=-1, cv=5, scoring='neg_mean_squared_error')

## Sa outlierima

Izdvajamo izlazna obelezja

In [34]:
y_Ca = data['Ca']
y_P  = data['P']
y_pH = data['pH']
y_SOC = data['SOC']
y_Sand = data['Sand']
X = data.drop(['Ca', 'P', 'pH', 'SOC', 'Sand'], axis=1)

In [35]:
from sklearn.model_selection import train_test_split

X_train_Ca, X_test_Ca, y_train_Ca, y_test_Ca = train_test_split(X, y_Ca, test_size=0.2, random_state=7)
X_train_P, X_test_P, y_train_P, y_test_P = train_test_split(X, y_P, test_size=0.2, random_state=7)
X_train_pH, X_test_pH, y_train_pH, y_test_pH = train_test_split(X, y_pH, test_size=0.2, random_state=7)
X_train_SOC, X_test_SOC, y_train_SOC, y_test_SOC = train_test_split(X, y_SOC, test_size=0.2, random_state=7)
X_train_Sand, X_test_Sand, y_train_Sand, y_test_Sand = train_test_split(X, y_Sand, test_size=0.2, random_state=7)

In [37]:
best_lin_reg_Ca = grid_lin_reg.fit(X_train_Ca, y_train_Ca).best_estimator_
best_lin_reg_P = grid_lin_reg.fit(X_train_P, y_train_P).best_estimator_
best_lin_reg_pH = grid_lin_reg.fit(X_train_pH, y_train_pH).best_estimator_
best_lin_reg_SOC = grid_lin_reg.fit(X_train_SOC, y_train_SOC).best_estimator_
best_lin_reg_Sand = grid_lin_reg.fit(X_train_Sand, y_train_Sand).best_estimator_

In [77]:
from sklearn.metrics import root_mean_squared_error

def calc_rmse(model_Ca, model_P, model_pH, model_SOC, model_Sand):
    y_RMSE_Ca = root_mean_squared_error(model_Ca.predict(X_test_Ca), y_test_Ca)
    y_RMSE_P = root_mean_squared_error(model_P.predict(X_test_P), y_test_P)
    y_RMSE_pH = root_mean_squared_error(model_pH.predict(X_test_pH), y_test_pH)
    y_RMSE_SOC = root_mean_squared_error(model_SOC.predict(X_test_SOC), y_test_SOC)
    y_RMSE_Sand = root_mean_squared_error(model_Sand.predict(X_test_Sand), y_test_Sand)

    print("Ca", y_RMSE_Ca)
    print("P", y_RMSE_P)
    print("pH", y_RMSE_pH)
    print("SOC", y_RMSE_SOC)
    print("Sand", y_RMSE_Sand, "\n")

    print("MCRMSE", (y_RMSE_Ca + y_RMSE_P + y_RMSE_pH + y_RMSE_SOC + y_RMSE_Sand)/5)

In [78]:
calc_rmse(best_lin_reg_Ca, best_lin_reg_P, best_lin_reg_pH, best_lin_reg_SOC, best_lin_reg_Sand)

Ca 0.3555080818818807
P 1.3395940064440024
pH 0.6578270496054428
SOC 0.5909757532033935
Sand 0.5982703433391132 

MCRMSE 0.7084350468947666


In [39]:
sorted_test = pd.read_csv('data/sorted_test.csv')
prediction_lin_reg = pd.DataFrame()
prediction_lin_reg['PIDN'] = sorted_test['PIDN']
sorted_test.drop('PIDN', inplace=True, axis=1)
sorted_test['Depth'] = sorted_test['Depth'].replace({'Topsoil' : 0 , 'Subsoil' : 1})

In [40]:
Ca = pd.DataFrame(best_lin_reg_Ca.predict(sorted_test))
P = pd.DataFrame(best_lin_reg_P.predict(sorted_test))
pH = pd.DataFrame(best_lin_reg_pH.predict(sorted_test))
SOC = pd.DataFrame(best_lin_reg_SOC.predict(sorted_test))
Sand = pd.DataFrame(best_lin_reg_Sand.predict(sorted_test))

In [41]:
prediction_lin_reg['Ca'] = Ca
prediction_lin_reg['P'] = P
prediction_lin_reg['pH'] = pH
prediction_lin_reg['SOC'] = SOC
prediction_lin_reg['Sand'] = Sand

In [42]:
prediction_lin_reg.head()

,PIDN,Ca,P,pH,SOC,Sand
0,09gt9UK5,-0.783310,-0.227541,-1.241273,-0.516128,1.516265
1,0BVvxJ6a,0.561543,-0.403230,1.534888,-0.062144,-1.348155
2,0KbdgApg,-0.367305,-0.039942,0.801908,0.349655,-0.757940
3,0MnuuduB,0.363112,0.555212,-0.536112,0.757542,-2.326632
4,0PcIsF9z,-0.807046,0.056620,-1.184271,-0.078686,-1.218557


In [44]:
prediction_lin_reg.to_csv('prediction_lin_reg.csv', index=False)

## Be outliera

In [74]:
y_Ca_c = data_clipped['Ca']
y_P_c  = data_clipped['P']
y_pH_c = data_clipped['pH']
y_SOC_c = data_clipped['SOC']
y_Sand_c = data_clipped['Sand']
X_c = data_clipped.drop(['Ca', 'P', 'pH', 'SOC', 'Sand'], axis=1)

In [75]:
X_train_Ca_c, X_test_Ca_c, y_train_Ca_c, y_test_Ca_c = train_test_split(X_c, y_Ca_c, test_size=0.2, random_state=7)
X_train_P_c, X_test_P_c, y_train_P_c, y_test_P_c = train_test_split(X_c, y_P_c, test_size=0.2, random_state=7)
X_train_pH_c, X_test_pH_c, y_train_pH_c, y_test_pH_c = train_test_split(X_c, y_pH_c, test_size=0.2, random_state=7)
X_train_SOC_c, X_test_SOC_c, y_train_SOC_c, y_test_SOC_c = train_test_split(X_c, y_SOC_c, test_size=0.2, random_state=7)
X_train_Sand_c, X_test_Sand_c, y_train_Sand_c, y_test_Sand_c = train_test_split(X_c, y_Sand_c, test_size=0.2, random_state=7)

In [76]:
best_lin_reg_Ca_c = grid_lin_reg.fit(X_train_Ca_c, y_train_Ca_c).best_estimator_
best_lin_reg_P_c = grid_lin_reg.fit(X_train_P_c, y_train_P_c).best_estimator_
best_lin_reg_pH_c = grid_lin_reg.fit(X_train_pH_c, y_train_pH_c).best_estimator_
best_lin_reg_SOC_c = grid_lin_reg.fit(X_train_SOC_c, y_train_SOC_c).best_estimator_
best_lin_reg_Sand_c = grid_lin_reg.fit(X_train_Sand_c, y_train_Sand_c).best_estimator_

In [79]:
calc_rmse(best_lin_reg_Ca_c, best_lin_reg_P_c, best_lin_reg_pH_c, best_lin_reg_SOC_c, best_lin_reg_Sand_c)

Ca 0.3937126939193669
P 1.4782738912890725
pH 0.605346686895232
SOC 0.6199822892573328
Sand 0.5863302253585929 

MCRMSE 0.7367291573439194


Zakljucujemo da linearna regresija trenirana na clippovanim podacima radi losije